This is where I test the two models together as a cascade instead of on
their own. YOLO runs first as a fast first pass to find candidate fall
regions, then Faster R-CNN re-checks those regions and gives the final
confidence score. The idea is that YOLO alone is fast but a bit trigger-happy,
and Faster R-CNN is slower but more careful, so chaining them should catch
more real falls without as many false alarms.

The first cell is just a quick check that a GPU is actually visible before
running anything heavier.

Quick sanity check that CUDA is available before doing anything else.

In [ ]:
import torch
print("CUDA available?", torch.cuda.is_available())
print("CUDA devices:", torch.cuda.device_count())


Everything for the cascade detector and its evaluation lives in this one
cell. Roughly, top to bottom:

- `OptimizedCascadeFallDetector` - loads both models and runs the two-stage
  cascade on an image or a batch of images.
- `SimpleYOLOValDataset` / `collect_validation_scores_fixed` - runs the
  cascade over the whole validation set and collects a fall-confidence
  score per image.
- The `plot_*` functions - confusion matrix, precision/recall vs.
  confidence, label distribution, all the charts I actually used in the
  results.
- `main_fixed()` - ties it all together: run the cascade over validation,
  pick the best confidence threshold, generate every plot, and save a
  written summary of the results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from ultralytics import YOLO
import cv2
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_curve,
    average_precision_score,
)
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.models import detection
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from PIL import Image
from pathlib import Path
import os
import json
import pickle
import yaml
import glob
from torch.utils.data import Dataset
from tqdm import tqdm
import time

# Configuration
lr = 0.01
path = f"C:/Users/bobe/Documents/Bobe/Paper/runs/train/Hybrid_noD_normal_lr.{lr}"
os.makedirs(path, exist_ok=True)

class OptimizedCascadeFallDetector:
    def __init__(self, yolo_path, rcnn_path,
                 device='cuda' if torch.cuda.is_available() else 'cpu',
                 batch_size=8):
        self.device = device
        self.batch_size = batch_size
        
        # Load YOLO
        self.yolo = YOLO(yolo_path)
        self.yolo.model = self.yolo.model.to(self.device)
        
        # Load Faster R-CNN
        self.rcnn = self._load_faster_rcnn(rcnn_path)
        
        # Optimized transform
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((640, 640)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
        
        print("  YOLO device:", next(self.yolo.model.parameters()).device)
        print("  RCNN device:", next(self.rcnn.parameters()).device)
        
    def _load_faster_rcnn(self, model_path):
        checkpoint = torch.load(model_path, map_location=self.device)
        state_dict = checkpoint.get('model_state_dict', checkpoint)
        
        num_classes = state_dict[
            'roi_heads.box_predictor.cls_score.weight'
        ].shape[0]
        print(f"Loading Faster R-CNN with {num_classes} classes")
        
        model = detection.fasterrcnn_resnet50_fpn(pretrained=False, num_classes=3)
        model.load_state_dict(state_dict)
        model.to(self.device)
        model.eval()
        return model

    def get_max_fall_confidence(self, img_path, yolo_conf=0.3, rcnn_conf=0.0, use_cascade=True):
        """Single image prediction - completely rewritten for reliability"""
        try:
            if use_cascade:
                # Step 1: YOLO detection
                yolo_results = self.yolo.predict(
                    source=img_path, 
                    conf=yolo_conf, 
                    device=self.device, 
                    verbose=False
                )
                
                for res in yolo_results:
                    data = res.boxes.data.clone()
                    data[:, 5] = data[:, 5] + 1
                    res.boxes.data = data
                
                # If no YOLO detections, return 0
                if len(yolo_results) == 0 or len(yolo_results[0].boxes) == 0:
                    return 0.0
            
            # Step 2: Faster R-CNN prediction
            img = cv2.imread(img_path)
            if img is None:
                print(f"Could not load image: {img_path}")
                return 0.0
            
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            tensor = self.transform(rgb).unsqueeze(0).to(self.device)
            
            with torch.no_grad():
                predictions = self.rcnn(tensor)
            
            # Extract fall confidences (assuming class 1 is fall)
            if len(predictions) == 0:
                return 0.0
                
            pred = predictions[0]
            if 'scores' not in pred or 'labels' not in pred:
                return 0.0
                
            BASE_RCNN_CONF = 0.3

            scores = pred['scores'].cpu().numpy()
            labels = pred['labels'].cpu().numpy() - 1  # –1 bg, 0 Fall, 1 NoFall

            # take only Fall scores above your base cutoff
            fall_scores = scores[(labels == 0) & (scores >= BASE_RCNN_CONF)]
            confidence = float(np.max(fall_scores)) if len(fall_scores) else 0.0
            
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            return 0.0

    def process_batch_images(self, image_paths, yolo_conf=0.3, rcnn_conf=0.0, use_cascade=True):
        """Process a batch of images with proper error handling"""
        confidences = []
        
        for img_path in image_paths:
            try:
                confidence = self.get_max_fall_confidence(img_path, yolo_conf, rcnn_conf, use_cascade)
                confidences.append(confidence)
            except Exception as e:
                print(f"Error processing {img_path}: {e}")
                confidences.append(0.0)
        
        return confidences

class SimpleYOLOValDataset(Dataset):
    """Simplified dataset that returns individual items properly"""
    def __init__(self, img_paths, lbl_paths, fall_cls=1):
        self.imgs = img_paths
        self.lbls = lbl_paths
        self.fall_cls = fall_cls
    
    def __len__(self):
        return len(self.imgs)
    
    def __getitem__(self, i):
        # Return image path and ground truth label
        gt = 0
        if os.path.isfile(self.lbls[i]):
            try:
                with open(self.lbls[i], 'r') as f:
                    for line in f:
                        line = line.strip()
                        if line and int(line.split()[0]) == self.fall_cls:
                            gt = 1
                            break
            except Exception as e:
                print(f"Error reading label file {self.lbls[i]}: {e}")
                gt = 0
        
        return self.imgs[i], gt

def collect_validation_scores_fixed(detector, dataset, batch_size=16):
    """Fixed validation score collection with proper batch processing"""
    y_true = []
    y_score = []
    
    print(f"Processing {len(dataset)} images in batches of {batch_size}...")
    
    # Process in batches
    for i in tqdm(range(0, len(dataset), batch_size), desc="Processing batches"):
        batch_end = min(i + batch_size, len(dataset))
        
        # Get batch data
        batch_paths = []
        batch_labels = []
        
        for j in range(i, batch_end):
            img_path, label = dataset[j]
            batch_paths.append(img_path)
            batch_labels.append(label)
        
        # Process batch
        try:
            confidences = detector.process_batch_images(batch_paths, use_cascade=False)
            
            # Add to results
            y_score.extend(confidences)
            y_true.extend(batch_labels)
            
        except Exception as e:
            print(f"Error processing batch {i//batch_size}: {e}")
            # Add zeros for failed batch
            y_score.extend([0.0] * len(batch_paths))
            y_true.extend(batch_labels)
        
        # Memory cleanup
        if i % (batch_size * 10) == 0:
            torch.cuda.empty_cache()
    
    print(f"Processed {len(y_true)} images")
    return np.array(y_true), np.array(y_score)

# ─── PLOTTING FUNCTIONS ─────────────────────────────────────────────────────────
def plot_confusion_matrix(cm, classes, normalize=False, title="Confusion Matrix", save_path=None):
    """Plot confusion matrix"""
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    
    ax.set(xticks=np.arange(cm.shape[1]),
           yticks=np.arange(cm.shape[0]),
           xticklabels=classes, yticklabels=classes,
           title=title,
           ylabel='True label',
           xlabel='Predicted label')
    
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt),
                   ha="center", va="center",
                   color="white" if cm[i, j] > thresh else "black")
    
    fig.tight_layout()
    
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Confusion matrix saved to {save_path}")
    
    return fig

def plot_pr_curve(y_true, y_score, save_path=None):
    """Plot Precision-Recall curve"""
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    ap_score = average_precision_score(y_true, y_score)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(recall, precision, linewidth=2, label=f'AP = {ap_score:.3f}')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title('Precision-Recall Curve')
    ax.legend()
    ax.grid(True)
    
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"PR curve saved to {save_path}")
    
    return fig

def plot_precision_vs_confidence(y_true, y_score, save_path=None):
    """Plot Precision vs Confidence Threshold"""
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    precision = precision[:-1]  # Remove last element to match threshold length
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(thresholds, precision, linewidth=2, color='blue', label='Precision')
    
    # Find max precision and its threshold
    max_precision_idx = np.argmax(precision)
    max_precision = precision[max_precision_idx]
    max_precision_threshold = thresholds[max_precision_idx]
    
    ax.axvline(x=max_precision_threshold, color='red', linestyle='--', alpha=0.7,
               label=f'Max Precision: {max_precision:.3f} @ {max_precision_threshold:.3f}')
    ax.scatter(max_precision_threshold, max_precision, color='red', s=100, zorder=5)
    
    ax.set_xlabel('Confidence Threshold')
    ax.set_ylabel('Precision')
    ax.set_title('Precision vs Confidence Threshold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Precision vs confidence plot saved to {save_path}")
    
    return fig, max_precision_threshold, max_precision

def plot_recall_vs_confidence(y_true, y_score, save_path=None):
    """Plot Recall vs Confidence Threshold"""
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    recall = recall[:-1]  # Remove last element to match threshold length
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(thresholds, recall, linewidth=2, color='green', label='Recall')
    
    # Find threshold for 90% recall (common benchmark)
    recall_90_idx = np.where(recall >= 0.9)[0]
    if len(recall_90_idx) > 0:
        recall_90_threshold = thresholds[recall_90_idx[-1]]  # Last threshold with 90% recall
        ax.axvline(x=recall_90_threshold, color='orange', linestyle='--', alpha=0.7,
                   label=f'90% Recall @ {recall_90_threshold:.3f}')
        ax.axhline(y=0.9, color='orange', linestyle=':', alpha=0.5)
    
    ax.set_xlabel('Confidence Threshold')
    ax.set_ylabel('Recall')
    ax.set_title('Recall vs Confidence Threshold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Recall vs confidence plot saved to {save_path}")
    
    return fig

def plot_confidence_analysis(y_true, y_score, save_path=None):
    """Plot precision, recall, and F1-score vs confidence threshold"""
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    
    precision = precision[:-1]
    recall = recall[:-1]
    
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.plot(thresholds, precision, label='Precision', linewidth=2, color='blue')
    ax.plot(thresholds, recall, label='Recall', linewidth=2, color='green')
    ax.plot(thresholds, f1_scores, label='F1-Score', linewidth=2, color='red')
    
    best_f1_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_f1_idx]
    best_f1 = f1_scores[best_f1_idx]
    
    ax.axvline(x=best_threshold, color='red', linestyle='--', alpha=0.7,
               label=f'Best F1 Threshold: {best_threshold:.3f}')
    ax.scatter(best_threshold, best_f1, color='red', s=100, zorder=5)
    
    ax.annotate(f'F1={best_f1:.3f}', 
                xy=(best_threshold, best_f1), 
                xytext=(best_threshold + 0.1, best_f1 + 0.05),
                arrowprops=dict(arrowstyle='->', color='red'),
                fontsize=10, color='red')
    
    ax.set_xlabel('Confidence Threshold')
    ax.set_ylabel('Score')
    ax.set_title('Performance Metrics vs Confidence Threshold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Confidence analysis plot saved to {save_path}")
    
    return fig, best_threshold, best_f1

def plot_label_distribution(y_true, class_names, save_path=None):
    """Plot distribution of labels"""
    unique, counts = np.unique(y_true, return_counts=True)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.bar([class_names[i] for i in unique], counts, 
                  color=['skyblue', 'lightcoral'])
    
    ax.set_ylabel('Count')
    ax.set_title('Label Distribution in Validation Set')
    
    # Add value labels on bars
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{count}', ha='center', va='bottom')
    
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Label distribution plot saved to {save_path}")
    
    return fig

def save_evaluation_results(y_true, y_score, best_threshold, save_dir=None):
    """Save evaluation results to files"""
    if save_dir is None:
        save_dir = f"{path}/evaluation_results"
    os.makedirs(save_dir, exist_ok=True)
    
    results = {
        'y_true': y_true.tolist(),
        'y_score': y_score.tolist(),
        'best_threshold': float(best_threshold),
    }
    
    results_path = os.path.join(save_dir, "evaluation_results.json")
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    
    pickle_path = os.path.join(save_dir, "evaluation_results.pkl")
    with open(pickle_path, 'wb') as f:
        pickle.dump(results, f)
    
    print(f"Evaluation results saved to:")
    print(f"JSON: {results_path}")
    print(f"Pickle: {pickle_path}")
    
    return results_path

# ─── MAIN EVALUATION SCRIPT ────────────────────────────────────────────────────
def main_fixed():
    """Completely fixed main function"""
    print("Starting fixed evaluation...")
    total_start = time.time()
    
    # Setup directories
    os.makedirs(f"{path}/evaluation_results", exist_ok=True)
    os.makedirs(f"{path}/evaluation_plots", exist_ok=True)

    # Initialize detector
    print("Loading models...")
    model_load_start = time.time()
    
    try:
        detector = OptimizedCascadeFallDetector(
            yolo_path=f"runs/train/YOLO_noD_normal_lr.{lr}/weights/best.pt",
            rcnn_path=f"runs/train/Faster_R-CNN_noD_normal_lr.{lr}/model_final.pt",
            batch_size=4
        )
    except Exception as e:
        print(f"Error loading models: {e}")
        return
    
    model_load_time = time.time() - model_load_start
    print(f"Model loading took {model_load_time:.2f} seconds")

    # Load dataset configuration
    try:
        cfg = yaml.safe_load(open('dataset_paper_new/data_nano.yaml'))
        val_img_dir = Path(cfg['val'])
        data_root = val_img_dir.parent.parent
        val_lbl_dir = data_root / 'labels' / 'val'

        # Get image and label paths
        imgs = sorted(glob.glob(str(val_img_dir / '*.png')))
        lbls = [str(val_lbl_dir / (Path(p).stem + '.txt')) for p in imgs]
        
        print(f"Found {len(imgs)} images for evaluation")

        # Determine fall class index
        names = cfg['names']
        if isinstance(names, dict):
            fall_idx = int(next(k for k, v in names.items() if v == 'Fall'))
        else:
            fall_idx = names.index('Fall')
            
    except Exception as e:
        print(f"Error loading dataset configuration: {e}")
        return

    # Create dataset (no DataLoader needed)
    val_ds = SimpleYOLOValDataset(imgs, lbls, fall_cls=fall_idx)
    print(f"Dataset created with {len(val_ds)} samples")

    # Collect validation scores
    print("Starting validation inference...")
    inference_start = time.time()
    
    try:
        y_true, y_score = collect_validation_scores_fixed(detector, val_ds, batch_size=8)
        zeros   = np.sum(y_score == 0.0)
        nonzero = len(y_score) - zeros
        print(f"Zeros: {zeros}, Non-zero: {nonzero}")
    except Exception as e:
        print(f"Error during inference: {e}")
        return
    
    inference_time = time.time() - inference_start
    print(f"Inference took {inference_time:.2f} seconds")
    print(f"Average time per image: {inference_time/len(y_true):.3f} seconds")

    print(f"Processed {len(y_true)} samples")
    print(f"Unique labels: {np.unique(y_true)}")
    print(f"Score range: {np.min(y_score):.3f} - {np.max(y_score):.3f}")
    
    # Skip analysis if no positive scores
    if np.max(y_score) == 0:
        print("Warning: All confidence scores are 0. Check your models and data.")
        return
    
    # Analysis
    analysis_start = time.time()
    
    try:
        # Find optimal threshold
        fig_conf, best_threshold, best_f1 = plot_confidence_analysis(
            y_true, y_score, 
            save_path=f"{path}/evaluation_plots/confidence_analysis.png"
        )
        print(f"Best F1-Score: {best_f1:.3f} at threshold: {best_threshold:.3f}")
        
        # Plot individual precision and recall vs confidence
        fig_prec, max_prec_thresh, max_prec = plot_precision_vs_confidence(
            y_true, y_score,
            save_path=f"{path}/evaluation_plots/P_curve.png"
        )
        
        fig_rec = plot_recall_vs_confidence(
            y_true, y_score,
            save_path=f"{path}/evaluation_plots/R_curve.png"
        )
        
        # Generate predictions with best threshold
        y_pred = (y_score >= best_threshold).astype(int)
        
        # Plot confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        class_names = ['No Fall', 'Fall']
        fig_cm = plot_confusion_matrix(
            cm, class_names, title="Confusion Matrix",
            save_path=f"{path}/evaluation_plots/confusion_matrix.png"
        )
        
        fig_cm_norm = plot_confusion_matrix(
            cm, class_names, normalize=True, 
            title="Normalized Confusion Matrix",
            save_path=f"{path}/evaluation_plots/confusion_matrix_normalized.png"
        )
        
        # Plot PR curve
        fig_pr = plot_pr_curve(
            y_true, y_score,
            save_path=f"{path}/evaluation_plots/PR_curve.png"
        )
        
        # Plot label distribution
        fig_dist = plot_label_distribution(
            y_true, class_names,
            save_path=f"{path}/evaluation_plots/labels.png"
        )
        
        # Calculate and print metrics
        tn, fp, fn, tp = cm.ravel()
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        
        print("\n" + "="*50)
        print("EVALUATION RESULTS")
        print("="*50)
        print(f"Accuracy:  {accuracy:.3f}")
        print(f"Precision: {precision:.3f}")
        print(f"Recall:    {recall:.3f}")
        print(f"F1-Score:  {f1:.3f}")
        print(f"AP Score:  {average_precision_score(y_true, y_score):.3f}")
        print(f"Best Threshold: {best_threshold:.3f}")
        print("="*50)
        
        # Save evaluation results
        save_evaluation_results(y_true, y_score, best_threshold)
        
        # Create summary report
        summary = {
            'metrics': {
                'accuracy': float(accuracy),
                'precision': float(precision),
                'recall': float(recall),
                'f1_score': float(f1),
                'ap_score': float(average_precision_score(y_true, y_score)),
                'best_threshold': float(best_threshold),
                'max_precision': float(max_prec),
                'max_precision_threshold': float(max_prec_thresh)
            },
            'confusion_matrix': cm.tolist(),
            'total_samples': len(y_true),
            'positive_samples': int(np.sum(y_true)),
            'negative_samples': int(len(y_true) - np.sum(y_true))
        }
        
        summary_path = f"{path}/evaluation_results/evaluation_summary.json"
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2)
        
        print(f"\nEvaluation summary saved to: {summary_path}")
        print("All plots saved to: evaluation_plots/")
        print("All results saved to: evaluation_results/")
        
    except Exception as e:
        print(f"Error during analysis: {e}")
        return
    
    total_time = time.time() - total_start
    print(f"\nTotal evaluation time: {total_time:.2f} seconds")
    print(f"Model loading: {model_load_time:.1f}s")
    print(f"Inference: {inference_time:.1f}s") 
    print(f"Analysis: {time.time() - analysis_start:.1f}s")
    
    plt.show()

if __name__ == "__main__":
    main_fixed()